# Train Mazesolver bot using PPO with curriculum learning

# Phase 1 - X,Y Tracking


In [3]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

# Import custom environment
# from envphase1 import Heuristic01Env
# from envphase1 import Heuristic01Env

# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/RL_workspace")

# Import PPO training module and exporter
from ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx

In [4]:
# Settings
MJCF_PATH = Path("/home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/RL_workspace/model/heuristic01_phase1.xml")
MJCF_PATH2 = Path("/home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/RL_workspace/model/heuristic01_phase2.xml")
SEED = 42
NUM_ENVS = 8         # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment

In [5]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "heuristic01",  # Name of the experiment
    env_id = "Mazesolver-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 64,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 64,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.00,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [9]:
def make_heuristic_bot_env(render, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = Heuristic01Env(
        mjcf_path    = MJCF_PATH,
        render_mode  = "human" if render else None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs mazesolver bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), kw=kwargs: make_heuristic_bot_env(render, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [10]:
env_factories = []
for i in range(NUM_ENVS):
    env_factories.append(lambda render=(i==0): make_heuristic_bot_env(render))

# Build vector of environments
envs = gym.vector.SyncVectorEnv(env_factories)

# Choo choo train
result = train(ppo_config, envs)

Run name: Mazesolver-v0__heuristic01__42__1789914485
TensorBoard: http://localhost:6006/#scalars&regexFilter=heuristic01
Reset: Goal set to (x=1.260, y=-0.509)
Reset: Goal set to (x=-0.500, y=-0.141)
Reset: Goal set to (x=0.046, y=-0.905)
Reset: Goal set to (x=1.243, y=0.225)
Reset: Goal set to (x=-1.225, y=-0.823)
Reset: Goal set to (x=0.619, y=0.377)
Reset: Goal set to (x=-0.711, y=-1.156)
Reset: Goal set to (x=1.277, y=0.082)
Reset: Goal set to (x=-0.141, y=0.612)
Reset: Goal set to (x=0.079, y=-0.917)
Reset: Goal set to (x=0.618, y=-1.002)
Reset: Goal set to (x=0.752, y=0.510)
Reset: Goal set to (x=0.982, y=-1.035)
Reset: Goal set to (x=-1.337, y=0.386)
Reset: Goal set to (x=-0.875, y=0.255)
Reset: Goal set to (x=0.933, y=0.962)
Reset: Goal set to (x=-0.416, y=0.847)
Reset: Goal set to (x=0.164, y=-0.907)
Reset: Goal set to (x=-1.089, y=0.540)
Reset: Goal set to (x=1.440, y=0.265)
Reset: Goal set to (x=0.531, y=0.190)
Reset: Goal set to (x=0.525, y=-1.353)
Reset: Goal set to (x=-0.

In [13]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Best model: runs/Mazesolver-v0__heuristic01__42__1789914485/best_model.cleanrl_model
Final model: runs/Mazesolver-v0__heuristic01__42__1789914485/heuristic01_final.cleanrl_model
Best mean return: 1273.16
Loaded best model (mean_return=1273.16)


In [6]:
import time
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym

# Import your environment
from envphase2 import Heuristic01Env

# Define your Agent network matching your training architecture
class Agent(nn.Module):
    def __init__(self, envs, ppo_config):
        super().__init__()
        
        # Dynamically calculate the observation size instead of hardcoding 8
        obs_size = np.array(envs.single_observation_space.shape).prod()
        
        self.critic = nn.Sequential(
            nn.Linear(obs_size, ppo_config.critic_hidden_size),
            nn.Tanh(),
            nn.Linear(ppo_config.critic_hidden_size, ppo_config.critic_hidden_size),
            nn.Tanh(),
            nn.Linear(ppo_config.critic_hidden_size, 1),
        )
        self.actor_mean = nn.Sequential(
            nn.Linear(obs_size, ppo_config.actor_hidden_size),
            nn.Tanh(),
            nn.Linear(ppo_config.actor_hidden_size, ppo_config.actor_hidden_size),
            nn.Tanh(),
            nn.Linear(ppo_config.actor_hidden_size, np.prod(envs.single_action_space.shape)),
        )
        self.actor_logstd = nn.Parameter(torch.zeros(1, np.prod(envs.single_action_space.shape)))

    def get_action_deterministic(self, x, deterministic=True):
        action_mean = self.actor_mean(x)
        if deterministic:
            return action_mean
        
        # If you want to test with exploration noise enabled during eval:
        action_std = torch.exp(self.actor_logstd.expand_as(action_mean))
        dist = torch.distributions.Normal(action_mean, action_std)
        return dist.sample()


def evaluate(model_path, config, num_episodes=5, render=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    render_mode = "human" if render else None
    env = Heuristic01Env(render_mode=render_mode)
    envs = gym.vector.SyncVectorEnv([lambda: env])

    # Pass config here so it matches the architecture used during training
    agent = Agent(envs, config).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    
    if isinstance(checkpoint, dict) and "model_weights" in checkpoint:
        agent.load_state_dict(checkpoint["model_weights"])
    else:
        agent.load_state_dict(checkpoint)
        
    agent.eval()
    print(f"Successfully loaded model from: {model_path}")
    
    # 3. Evaluation Loop
    for episode in range(num_episodes):
        obs, _ = envs.reset()
        done = False
        total_reward = 0.0
        steps = 0

        while not done:
            with torch.no_grad():
                obs_tensor = torch.Tensor(obs).to(device)
                # Set deterministic=False here to see if exploration noise keeps it balanced like training!
                action = agent.get_action_deterministic(obs_tensor, deterministic=True)

            obs, reward, terminations, truncations, _ = envs.step(action.cpu().numpy())
            done = np.logical_or(terminations, truncations)[0]
            
            total_reward += reward[0]
            steps += 1

            if render:
                env.render()
                time.sleep(0.001)  # Smooth render frame rate

        print(f"Episode {episode + 1}: Total Reward = {total_reward:.2f} | Steps = {steps}")

    env.close()

if __name__ == "__main__":
    # MODEL_PATH = "/home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/RL_workspace/scripts/runs/Mazesolver-v0__heuristic01__42__1789909515/best_model.cleanrl_model"
    MODEL_PATH = "/home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/RL_workspace/scripts/runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789921721/best_model.cleanrl_model"

    # Note: Ensure ppo_config is defined in the namespace before calling this 
    evaluate(MODEL_PATH, config=ppo_config, num_episodes=20, render=True)

Successfully loaded model from: /home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/RL_workspace/scripts/runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789921721/best_model.cleanrl_model


/home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/venv/lib/python3.12/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


Episode 1: Total Reward = -213.57 | Steps = 608
Episode 2: Total Reward = 768.28 | Steps = 2946
Episode 3: Total Reward = 892.57 | Steps = 2936
Episode 4: Total Reward = 754.24 | Steps = 2949
Episode 5: Total Reward = 892.69 | Steps = 2936
Episode 6: Total Reward = 981.90 | Steps = 2934
Episode 7: Total Reward = -208.21 | Steps = 539
Episode 8: Total Reward = 959.36 | Steps = 2934
Episode 9: Total Reward = 837.16 | Steps = 2938
Episode 10: Total Reward = 758.46 | Steps = 2948
Episode 11: Total Reward = 755.78 | Steps = 2948
Episode 12: Total Reward = 781.48 | Steps = 2943
Episode 13: Total Reward = 782.41 | Steps = 2943
Episode 14: Total Reward = -207.66 | Steps = 528
Episode 15: Total Reward = 849.25 | Steps = 2937
Episode 16: Total Reward = 849.83 | Steps = 2937
Episode 17: Total Reward = 925.44 | Steps = 2935
Episode 18: Total Reward = 774.30 | Steps = 2945
Episode 19: Total Reward = 1016.09 | Steps = 2933
Episode 20: Total Reward = 762.00 | Steps = 2947


In [14]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Best model: runs/Mazesolver-v0__heuristic01__42__1789914485/best_model.cleanrl_model
Final model: runs/Mazesolver-v0__heuristic01__42__1789914485/heuristic01_final.cleanrl_model
Best mean return: 1273.16
Loaded best model (mean_return=1273.16)


In [15]:
# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Exported charts_metrics.csv (5 metrics, 1292 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


In [16]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3


In [17]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

ModuleNotFoundError: No module named 'onnxscript'

# Phase 2 - Wall Following


In [4]:
from envphase2 import Heuristic01Env
model_path = "/home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/RL_workspace/scripts/runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789921721/best_model.cleanrl_model"

In [5]:
def load_agent(envs, model_path):
    """
    For debugging only! Use this to load a previously trained model to skip previous phases. Note 
    that you will still need to run the cells in each prior phase that update the experiment name 
    and environment.
    """
    from ppo_trainer import Agent, TrainResult

    # Make sure model_path is a Path
    model_path = Path(model_path)
    
    # Load agent from previous run
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = Agent(envs, ppo_config).to(device)
    agent.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    agent.eval()
    print(f"Loaded model from {model_path}")
    
    # Wrap agent in a dummy result
    return TrainResult(
        agent = agent,
        checkpoint_dir = model_path.parent,
        best_model_path = model_path,
        final_model_path = None,
        best_mean_return = 0,
    )

In [6]:
def make_heuristic_bot_env(render, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = Heuristic01Env(
        mjcf_path    = MJCF_PATH2,
        render_mode  = "human" if render else None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs mazesolver bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), kw=kwargs: make_heuristic_bot_env(render, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [7]:
ppo_config.exp_name = "mazesolver-bot-phase-2"
envs = make_envs(NUM_ENVS)
loaded_result = load_agent(envs,model_path)
agent = loaded_result.agent

Loaded model from /home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/RL_workspace/scripts/runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789921721/best_model.cleanrl_model


In [8]:
result = train(ppo_config, envs=envs, agent=agent)
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: Mazesolver-v0__mazesolver-bot-phase-2__42__1789922701
TensorBoard: http://localhost:6006/#scalars&regexFilter=mazesolver-bot-phase-2


/home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/venv/lib/python3.12/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


Checkpoint saved to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789922701/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=471.98) to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789922701/best_model.cleanrl_model
Checkpoint saved to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789922701/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=867.76) to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789922701/best_model.cleanrl_model
Checkpoint saved to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789922701/checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=1122.30) to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789922701/best_model.cleanrl_model
Checkpoint saved to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789922701/checkpoint_iter0200.cleanrl_model
Final model saved to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789922701/mazesolver-bot-phase-2_final.cleanrl_model
Best model mean return: 1122.30, save

In [16]:
ppo_config.ent_coef = 0.005  # Forces exploration and prevents action collapse
ppo_config.total_timesteps = 1_000_000  # You only need a fraction of the steps (e.g., 1M instead of 4M) for fine-tuning!

# 2. Load your current best model checkpoint as the starting point
loaded_result = load_agent(envs, model_path)
agent = loaded_result.agent
result = train(ppo_config, envs=envs, agent=agent)

Loaded model from /home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/RL_workspace/scripts/runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789920501/best_model.cleanrl_model
Run name: Mazesolver-v0__mazesolver-bot-phase-2__42__1789921721
TensorBoard: http://localhost:6006/#scalars&regexFilter=mazesolver-bot-phase-2


/home/mohit/STM32CubeIDE/workspace_2.2.0/Heuristic-01/venv/lib/python3.12/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


Checkpoint saved to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789921721/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=703.34) to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789921721/best_model.cleanrl_model
Final model saved to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789921721/mazesolver-bot-phase-2_final.cleanrl_model
Best model mean return: 703.34, saved to runs/Mazesolver-v0__mazesolver-bot-phase-2__42__1789921721/best_model.cleanrl_model
